# Consultas PETRVS — Denodo via Jupyter

Conecta ao banco `petrvs_icmbio` via JDBC direto.  
Use a função `run_query(sql)` para executar qualquer consulta e receber um DataFrame do pandas.

> **Antes de usar pela primeira vez:** copie o arquivo `.env.example` como `.env` na raiz do projeto e preencha com suas credenciais.  
> Veja o passo a passo completo em `docs/10-jupyter-guia-iniciantes.md`.

## Pré-requisito único — instalar python-dotenv

Execute esta célula **uma única vez** no primeiro uso. Nas próximas sessões, pule direto para a Seção 1.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "python-dotenv"],
    capture_output=True, text=True
)
linhas = [l for l in result.stdout.splitlines() if l.strip()]
print(linhas[-1] if linhas else "python-dotenv já instalado.")

## 1. Configuração da conexão

> **Sempre que reabrir o VS Code**, execute as duas células abaixo antes de fazer qualquer consulta.

In [ ]:
import os
from dotenv import load_dotenv
import jpype
import pandas as pd

# Carrega as credenciais do arquivo .env (fica apenas no seu computador — nunca vai ao GitHub)
load_dotenv()

JAVA_HOME    = os.getenv("JAVA_HOME",         "C:/Program Files/DBeaver/jre")
DRIVER_JAR   = os.getenv("DENODO_DRIVER_PATH", "")
DRIVER_CLASS = "com.denodo.vdp.jdbc.Driver"
JDBC_URL     = (
    "jdbc:denodo://"
    + os.getenv("DENODO_HOST",     "denodo-pgd.dataprev.gov.br")
    + ":"
    + os.getenv("DENODO_PORT",     "443")
    + "/"
    + os.getenv("DENODO_DATABASE", "petrvs_icmbio")
)
DB_USER = os.getenv("DENODO_USER",     "")
DB_PASS = os.getenv("DENODO_PASSWORD", "")

# Valida se as credenciais foram carregadas
if not DB_USER or not DB_PASS:
    print("ATENÇÃO: Credenciais não encontradas.")
    print("Verifique se o arquivo .env existe na raiz do projeto e está preenchido.")
    print("Copie o arquivo .env.example como .env e preencha com suas credenciais.")
elif not os.path.isfile(DRIVER_JAR):
    print("ATENÇÃO: Driver Denodo não encontrado em:")
    print(f"  {DRIVER_JAR}")
    print("Verifique o caminho DENODO_DRIVER_PATH no arquivo .env.")
else:
    os.environ["JAVA_HOME"] = JAVA_HOME
    if not jpype.isJVMStarted():
        jpype.startJVM(
            jpype.getDefaultJVMPath(),
            f"-Djava.class.path={DRIVER_JAR}",
            convertStrings=True,
        )
        print("JVM iniciada.")
    else:
        print("JVM já estava ativa.")
    print("Configuração OK.")

In [ ]:
def run_query(sql: str) -> pd.DataFrame:
    Properties    = jpype.JClass("java.util.Properties")
    DriverManager = jpype.JClass("java.sql.DriverManager")
    jpype.JClass(DRIVER_CLASS)

    props = Properties()
    props.put("user",     DB_USER)
    props.put("password", DB_PASS)

    conn = DriverManager.getConnection(JDBC_URL, props)

    try:
        conn.setCatalog("petrvs_icmbio")
    except Exception:
        pass

    try:
        stmt = conn.createStatement()
        try:
            rs   = stmt.executeQuery(sql)
            meta = rs.getMetaData()
            n    = meta.getColumnCount()
            cols = [meta.getColumnName(i + 1) for i in range(n)]
            rows = []
            while rs.next():
                rows.append([rs.getObject(i + 1) for i in range(n)])
            return pd.DataFrame(rows, columns=cols)
        finally:
            try: stmt.close()
            except Exception: pass
    finally:
        try: conn.close()
        except Exception: pass

print("Função run_query() pronta.")

## 2. Teste de conexão

In [ ]:
df_teste = run_query("""
    SELECT COUNT(*) AS total_entregas_ativas
    FROM petrvs_icmbio_planos_entregas_entregas
    WHERE deleted_at IS NULL
""")
df_teste

## 3. Exemplos de indicadores

Ajuste `data_inicio` e `data_fim` conforme o ciclo desejado.

### I02 — Taxa de cumprimento das entregas por unidade

In [ ]:
sql_i02 = """
WITH parametros AS (
    SELECT
        CAST('2025-01-01' AS DATE) AS data_inicio,
        CAST('2025-12-31' AS DATE) AS data_fim,
        0                          AS incluir_excluidos
),
entregas AS (
    SELECT
        pe.unidade_id,
        u.sigla                                                    AS unidade,
        pee.id                                                     AS entrega_id,
        COALESCE(
            NULLIF(TRIM(pee.descricao),''),
            NULLIF(TRIM(pee.descricao_entrega),'')
        )                                                          AS nome_entrega,
        pee.progresso_esperado,
        pee.progresso_realizado,
        CASE
            WHEN pee.progresso_realizado >= pee.progresso_esperado THEN 1
            ELSE 0
        END                                                        AS concluida
    FROM petrvs_icmbio_planos_entregas_entregas pee
    JOIN petrvs_icmbio_planos_entregas pe
        ON pee.plano_entrega_id = pe.id
    JOIN petrvs_icmbio_unidades u
        ON pe.unidade_id = u.id
    CROSS JOIN parametros p
    WHERE (p.incluir_excluidos = 1 OR pee.deleted_at IS NULL)
      AND (p.incluir_excluidos = 1 OR pe.deleted_at  IS NULL)
      AND pe.data_inicio >= p.data_inicio
      AND pe.data_fim    <= p.data_fim
)
SELECT
    unidade,
    COUNT(*)                                                       AS total_entregas,
    SUM(concluida)                                                 AS entregas_concluidas,
    ROUND(100.0 * SUM(concluida) / NULLIF(COUNT(*), 0), 1)        AS taxa_cumprimento_perc
FROM entregas
GROUP BY unidade_id, unidade
ORDER BY taxa_cumprimento_perc DESC
"""

df_i02 = run_query(sql_i02)
df_i02

### Consulta livre

Cole aqui qualquer query do manual técnico (`docs/06.x-ixxx.md`).  
Lembre-se: nomes de tabela usam o prefixo `petrvs_icmbio_` e datas usam `CAST('AAAA-MM-DD' AS DATE)`.

In [ ]:
sql_livre = """
SELECT 1 AS ok
"""

df_livre = run_query(sql_livre)
df_livre

## 4. Exportar resultado para CSV

In [ ]:
# Substitua df_i02 pelo DataFrame que deseja exportar (df_i02, df_livre, df_teste)
output_path = r"Tabelas CSV\i02_taxa_cumprimento.csv"
df_i02.to_csv(output_path, index=False, sep=";", encoding="utf-8-sig")
print(f"Exportado: {output_path}")